
## 1 - Environment Setup & Imports

In [ ]:
# Test Code for PyTorch and CUDA setup (Not Needed for the main project, but useful for debugging)
import time
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Matrix size
N = 10000

# -------------------------
# CPU Benchmark
# -------------------------
print("\n--- CPU Benchmark ---")
x_cpu = torch.randn(N, N)
y_cpu = torch.randn(N, N)

start = time.time()
z_cpu = x_cpu @ y_cpu
end = time.time()

print(f"CPU Time: {end - start:.4f} seconds")
print("z_cpu shape:", z_cpu.shape)
print("z_cpu device:", z_cpu.device)

# -------------------------
# GPU Benchmark (only if CUDA works)
# -------------------------
if torch.cuda.is_available():
    print("\n--- GPU Benchmark ---")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

    # Create tensors directly on GPU
    x_gpu = torch.randn(N, N, device="cuda")
    y_gpu = torch.randn(N, N, device="cuda")

    # 1) Warm-up run (important!)
    # This removes first-time overhead like kernel compilation, caching, etc.
    _ = x_gpu @ y_gpu
    torch.cuda.synchronize()

    # 2) Actual timed run
    start = time.time()

    z_gpu = x_gpu @ y_gpu
    torch.cuda.synchronize()  # wait for GPU to finish

    end = time.time()

    print(f"GPU Time: {end - start:.4f} seconds")
    print("z_gpu shape:", z_gpu.shape)
    print("z_gpu device:", z_gpu.device)

else:
    print("\nCUDA not detected. GPU benchmark skipped.")


In [1]:
# Standard library
import os
import re
import json
import random
import string
import pickle
import warnings
warnings.filterwarnings('ignore')

# Numerical / Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
# https://matplotlib.org/stable/users/explain/customizing.html
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F7F4',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE = {'train': '#534AB7', 'val': '#D85A30', 'test': '#1D9E75'}

# NLP
import nltk
# https://www.gutenberg.org/
nltk.download('gutenberg', quiet=True)
nltk.download('punkt',     quiet=True)
from nltk.corpus import gutenberg

# ML / Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks as keras_callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import top_k_accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# Todo: GPU not getting detected. Fix this before running the main project. For now, we will proceed with CPU training, but it will be much slower.
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}  (CPU training will be used)')
print('All imports successful.')

TensorFlow version: 2.21.0
GPU available: False  (CPU training will be used)
All imports successful.


In [2]:
# Todo: Remove after debuggibg. Additional GPU check using TensorFlow (since PyTorch check was done earlier)
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPU devices:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.21.0
Built with CUDA: False
GPU devices: []


---
## 2 — Data Acquisition & Corpus Loading

In [3]:
# Load Shakespeare plays from NLTK Gutenberg corpus
# Todo :: 3 Shakespeare plays (Todo: Add more texts if needed)
# https://www.gutenberg.org/cache/epub/100/pg100.txt
shakespeare_files = [f for f in gutenberg.fileids() if 'shakespeare' in f]
print('Shakespeare files available in NLTK Gutenberg:')

raw_texts = []
for fname in shakespeare_files:
    raw = gutenberg.raw(fname)
    raw_texts.append(raw)
    print(f'  {fname:<30} →  {len(raw):,} chars')

# Combine all three plays into one corpus string
CORPUS_RAW = '\n\n'.join(raw_texts)
print(f'\nCombined corpus length : {len(CORPUS_RAW):,} characters')
print('First 500 characters:\n', CORPUS_RAW[:500])

Shakespeare files available in NLTK Gutenberg:
  shakespeare-caesar.txt         →  112,310 chars
  shakespeare-hamlet.txt         →  162,881 chars
  shakespeare-macbeth.txt        →  100,351 chars

Combined corpus length : 375,546 characters
First 500 characters:
 [The Tragedie of Julius Caesar by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Flauius, Murellus, and certaine Commoners ouer the Stage.

  Flauius. Hence: home you idle Creatures, get you home:
Is this a Holiday? What, know you not
(Being Mechanicall) you ought not walke
Vpon a labouring day, without the signe
Of your Profession? Speake, what Trade art thou?
  Car. Why Sir, a Carpenter

   Mur. Where is thy Leather Apron, and thy Rule?
What dost thou with thy best Apparrell on


---
## 3 — Text Preprocessing & Tokenisation

In [4]:
# Preprocessing function
def clean_text(text: str) -> str:
    """Normalise Shakespeare text for language modelling."""
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)    
    text = re.sub(r"[^a-z\s.,!?;:'\-]", ' ', text)    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

CORPUS_CLEAN = clean_text(CORPUS_RAW)

# Keras Tokenizer
# https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer
tokenizer = Tokenizer(
    num_words=None,
    oov_token='<OOV>',
    lower=True,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts([CORPUS_CLEAN])

# Vocabulary statistics
word_index  = tokenizer.word_index
index_word  = tokenizer.index_word
VOCAB_SIZE  = len(word_index) + 1
TOTAL_WORDS = sum(tokenizer.word_counts.values())

print('Preprocessing')
print(f'  Raw corpus length   : {len(CORPUS_RAW):,} chars')
print(f'  Cleaned length      : {len(CORPUS_CLEAN):,} chars')
print(f'  Total word tokens   : {TOTAL_WORDS:,}')
print(f'  Vocabulary size     : {VOCAB_SIZE:,}   (words appearing ≥ 2 times)')
print(f'  OOV token idx       : {word_index["<OOV>"]}  (<OOV>)')
print(f'\nSample cleaned text (first 300 chars):\n', CORPUS_CLEAN[:300])
print(f'\nSample word index entries (first 10):', list(tokenizer.word_index.items())[:10])

Preprocessing
  Raw corpus length   : 375,546 chars
  Cleaned length      : 365,661 chars
  Total word tokens   : 67,876
  Vocabulary size     : 7,793   (words appearing ≥ 2 times)
  OOV token idx       : 1  (<OOV>)

Sample cleaned text (first 300 chars):
 actus primus. scoena prima. enter flauius, murellus, and certaine commoners ouer the stage. flauius. hence: home you idle creatures, get you home: is this a holiday? what, know you not being mechanicall you ought not walke vpon a labouring day, without the signe of your profession? speake, what trad

Sample word index entries (first 10): [('<OOV>', 1), ('the', 2), ('and', 3), ('to', 4), ('i', 5), ('of', 6), ('you', 7), ('a', 8), ('my', 9), ('that', 10)]


---
## Cell 4 — Sequence Generation & Dataset Construction

In [5]:
# Hyperparameters => CPU => GPU (Gaming -> Polygons -> Arrays) => TPU (Nvidia)
SEQ_LEN = 40
STRIDE   = 3

# Convert entire corpus to a flat list of integer token Id
token_ids = tokenizer.texts_to_sequences([CORPUS_CLEAN])[0]

# Build (X, y) pairs using sliding window
X_seqs, y_seqs = [], []
for i in range(0, len(token_ids) - SEQ_LEN, STRIDE):
    X_seqs.append(token_ids[i : i + SEQ_LEN])
    y_seqs.append(token_ids[i + SEQ_LEN])

X_all = np.array(X_seqs, dtype=np.int32) 
y_all = np.array(y_seqs, dtype=np.int32)

# OHE
y_cat = to_categorical(y_all, num_classes=VOCAB_SIZE)

# ─── Train / Val / Test split  (80 / 10 / 10) ────────────────────────────────
# First split out 20% for val+test, then split that equally
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_cat, test_size=0.20, random_state=SEED, shuffle=True
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, shuffle=True
)

# Also keep integer labels for top-k accuracy evaluation later
_, y_int_tmp = train_test_split(y_all, test_size=0.20, random_state=SEED, shuffle=True)
y_val_int, y_test_int = train_test_split(y_int_tmp, test_size=0.50, random_state=SEED, shuffle=True)

print('── Dataset construction ──────────────────────────────────')
print(f'  Sequence length     : {SEQ_LEN}  tokens (input)')
print(f'  Sliding stride      : {STRIDE}   tokens')
print(f'  Total sequences     : {len(X_all):,}')
print(f'\n── Train / Val / Test split ──────────────────────────────')
print(f'  Training samples    : {len(X_train):,}  (80.0%)')
print(f'  Validation samples  : {len(X_val):,}   (10.0%)')
print(f'  Test samples        : {len(X_test):,}   (10.0%)')
print(f'\n  X_train shape       : {X_train.shape}')
print(f'  y_train shape       : {y_train.shape}')
print(f'  X_val shape         : {X_val.shape}')
print(f'  X_test shape        : {X_test.shape}')
print(f'\nMemory footprint (training X): {X_train.nbytes / 1e6:.1f} MB')

── Dataset construction ──────────────────────────────────
  Sequence length     : 40  tokens (input)
  Sliding stride      : 3   tokens
  Total sequences     : 22,612

── Train / Val / Test split ──────────────────────────────
  Training samples    : 18,089  (80.0%)
  Validation samples  : 2,261   (10.0%)
  Test samples        : 2,262   (10.0%)

  X_train shape       : (18089, 40)
  y_train shape       : (18089, 7793)
  X_val shape         : (2261, 40)
  X_test shape        : (2262, 40)

Memory footprint (training X): 2.9 MB


---
## Cell 5 — Model Architecture

In [6]:
# https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/
# https://www.tensorflow.org/text/tutorials/text_generation
# https://karpathy.github.io/2015/05/21/rnn-effectiveness/

# Hyperparameters
EMBED_DIM   = 128   # Embedding dimension (Dense vector) https://medium.com/@yasindusanjeewa8/dense-vectors-in-natural-language-processing-06818dff5cd7
LSTM1_UNITS = 256   # 1st Bi-LSTM hidden units
LSTM2_UNITS = 128   # 2nd Bi-LSTM hidden units
LSTM3_UNITS = 64    # 3rd Bi-LSTM hidden units
MHA_HEADS   = 4     # Todo :: Check with 8 :: Multi-head attention: number of heads. 4 parallel attention
# https://medium.com/@zhonghong9998/attention-mechanisms-in-deep-learning-enhancing-model-performance-32a91006092a
# https://www.geeksforgeeks.org/artificial-intelligence/ml-attention-mechanism/
MHA_KEY_DIM = 32    # Key dimensionality per head. 32-dimensional K,Q vectors. 
DENSE_UNITS = 256   # Penultimate dense layer
DROP_EMB    = 0.30  # SpatialDropout on embeddings
DROP_LSTM   = 0.30  # Dropout between LSTM stacks. ToDo :: Need to reduce/increase
DROP_DENSE  = 0.40  # Dropout before output. ToDo :: Need to reduce/increase
L2_REG      = 1e-5  # L2 weight regularisation

# Build model using Keras Functional API
# https://keras.io/api/models/model/
def build_model(vocab_size: int, seq_len: int) -> keras.Model:
    
    # https://keras.io/api/layers/core_layers/input/
    inputs = keras.Input(shape=(seq_len,), name='token_ids')

    # Embedding layer: maps token IDs -> dense vectors
    # https://keras.io/api/layers/core_layers/embedding/
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        embeddings_regularizer=regularizers.l2(L2_REG),
        name='embedding'
    )(inputs)

    # SpatialDropout1D
    # https://keras.io/api/layers/regularization_layers/spatial_dropout1d/
    # https://stackoverflow.com/questions/50393666/how-to-understand-spatialdropout1d-and-when-to-use-it
    x = layers.SpatialDropout1D(DROP_EMB, name='spatial_dropout')(x)

    # Bi-LSTM layer 1
    # return_sequences=True
    # https://keras.io/api/layers/recurrent_layers/lstm/
    x = layers.Bidirectional(
        layers.LSTM(
            LSTM1_UNITS,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(L2_REG),
            recurrent_dropout=0.0,   # set to 0 for GPU compatibility ToDo :: Check GPU 
            name='lstm_1'
        ),
        name='bidirectional'
    )(x)
    # Stabilizes activations during training.
    x = layers.BatchNormalization(momentum=0.9, name='bn_1')(x)
    x = layers.Dropout(DROP_LSTM, name='drop_1')(x)

    # Bi-LSTM layer 2
    x = layers.Bidirectional(
        layers.LSTM(
            LSTM2_UNITS,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(L2_REG),
            name='lstm_2'
        ),
        name='bidirectional_1'
    )(x)
    x = layers.BatchNormalization(momentum=0.9, name='bn_2')(x)
    x = layers.Dropout(DROP_LSTM, name='drop_2')(x)

    # Bi-LSTM layer 3
    lstm3_out = layers.Bidirectional(
        layers.LSTM(
            LSTM3_UNITS,
            return_sequences=True,     # keep sequence dim for attention
            kernel_regularizer=regularizers.l2(L2_REG),
            name='lstm_3'
        ),
        name='bidirectional_2'
    )(x)
    # lstm3_out shape: (batch, seq_len, 128)

    # Multi-Head Self-Attention
    # https://stackoverflow.com/questions/75590491/difference-between-multiheadattention-and-attention-layer-in-tensorflow
    # https://keras.io/api/layers/attention_layers/multi_head_attention/
    mha_out = layers.MultiHeadAttention(
        num_heads=MHA_HEADS,
        key_dim=MHA_KEY_DIM,
        dropout=0.1,
        name='multi_head_attention'
    )(lstm3_out, lstm3_out)   # self-attention: (query=lstm3, value=lstm3)

    # Residual connection = original LSTM output + attention output
    x = layers.Add(name='residual_add')([lstm3_out, mha_out])
    x = layers.LayerNormalization(epsilon=1e-6, name='layer_norm')(x)

    # Compress sequence dimension using Global Average Pooling
    # https://keras.io/api/layers/pooling_layers/global_average_pooling1d/
    x = layers.GlobalAveragePooling1D(name='gap')(x)

    # Dense classification
    x = layers.Dense(
        DENSE_UNITS,
        activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='dense_hidden'
    )(x)
    x = layers.Dropout(DROP_DENSE, name='drop_dense')(x)

    # probability over full vocabulary
    outputs = layers.Dense(
        vocab_size,
        activation='softmax',
        name='output_softmax'
    )(x)

    model = keras.Model(inputs, outputs, name='NextWordPredictor_BiLSTM_MHA')
    return model

# Build and display the model
model = build_model(VOCAB_SIZE, SEQ_LEN)
model.summary()

Model: "NextWordPredictor_BiLSTM_MHA"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 40, 128)   │    997,504 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 40, 128)   │          0 │ embedding[0][0]   │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 40, 512)   │    788,480 │ spatial_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 40, 512)   │      2,048 │ bidirectional[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_1 (Dropout)    │ (None, 40, 512)   │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 40, 256)   │    656,384 │ drop_1[0][0]      │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 40, 256)   │      1,024 │ bidirectional_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_2 (Dropout)    │ (None, 40, 256)   │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 40, 128)   │    164,352 │ drop_2[0][0]      │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 40, 128)   │     66,048 │ bidirectional_2[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_add (Add)  │ (None, 40, 128)   │          0 │ bidirectional_2[… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm          │ (None, 40, 128)   │        256 │ residual_add[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 128)       │          0 │ layer_norm[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 256)       │     33,024 │ gap[0][0]         │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_dense          │ (None, 256)       │          0 │ dense_hidden[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_softmax      │ (None, 7793)      │  2,002,801 │ drop_dense[0][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 4,711,921 (17.97 MB)

 Trainable params: 4,710,385 (17.97 MB)

 Non-trainable params: 1,536 (6.00 KB)

---
## Cell 6 — Perplexity & Model Compilation

---
## Cell 7 — Training

---
## Cell 8 — Visualisations

---
## Cell 9 — Test Set Evaluation

---
## Cell 10 — Text Generation

---
## Cell 11 — Prediction 

---
## Cell 12 — Save Artefacts & Summary